## Setup

In [0]:
pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
# mlflow.autolog()

## Get the conf from the local conf file
model_config = mlflow.models.ModelConfig(development_config="../conf/chapter05_conf.yml")

retriever_configs = model_config.get("retriever_configs")

## Single Retriever

In [0]:
from typing import List, Dict, Optional, Any
from databricks.vector_search.client import VectorSearchClient

from mlflow.entities import SpanType, Document


class VectorSearchWrapper:
    def __init__(self, retriever_config: Dict):
        """
        Initialize the VectorSearchWrapper with a single retriever config.

        :param retriever_config: Dictionary containing the retriever config.
            Example:
            {
              'endpoint_name': 'vs_endpoint',
              'index_name': 'workspace.unity_air.faq_index',
              'columns': ['id', 'question', 'answer', 'search_text'],
              'k': 3,
              'retriever_schema': {
                  'primary_key': 'id',
                  'text_column': 'search_text',
                  'doc_uri': 'id',
                  'name': 'unity_air_faq_vs_index'
              }
            }
        """
        self.vsc = VectorSearchClient()
        self.retriever_cfg = retriever_config

        # Create the index handle
        self.index = self.vsc.get_index(
            endpoint_name=retriever_config["endpoint_name"],
            index_name=retriever_config["index_name"],
        )
    
    @mlflow.trace(span_type=SpanType.RETRIEVER, name="single_retriever_search", attributes={"vs_type": "databricks_vector_search"})
    def search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ):
        """
        Perform a similarity search against the retriever's index.

        :param query_text: The text query to search for.
        :param columns: Optional override for columns to return.
        :param filters: Optional filters to apply.
        :param num_results: Optional override for number of results (k).
        :return: Search results from the vector index.
        """
        
        mlflow.update_current_trace(tags={"vs_endpoint_name": self.retriever_cfg["endpoint_name"]})
        mlflow.update_current_trace(tags={"vs_index_name": self.retriever_cfg["index_name"]})

        span = mlflow.get_current_active_span()
        span.set_attribute("filters", filters or self.retriever_cfg.get("filters", {}))
        span.set_attribute("retriever_k", num_results or self.retriever_cfg.get("k", 5))
                             
        return self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )
    
    @mlflow.trace(span_type=SpanType.PARSER, name="Parse_Search_Result")
    def parse_search_result(self, search_result):
        columns = search_result['manifest']["columns"]
        data_array = search_result.get("result").get("data_array")

        retriever_schema = self.retriever_cfg['retriever_schema']

        mapped_result = {}
        output_list = []

        if len(data_array) > 0:
            for data in data_array:
                for column, column_value in zip(columns, data):
                    mapped_result[column['name']] = column_value
                
                metadata = {'score': mapped_result['score']}
                doc = Document(
                    page_content = mapped_result[retriever_schema['text_column']],
                    metadata = metadata,
                    id = mapped_result[retriever_schema['primary_key']]
                )
                output_list.append(doc)

        return output_list
    
    @mlflow.trace(span_type=SpanType.RETRIEVER)
    def refined_search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
        ):

        search_results = self.search(
            query_text=query_text,
            columns=columns,
            filters=filters,
            num_results=num_results,
        )

        parsed_results = self.parse_search_result(search_result=search_results)

        return parsed_results

In [0]:
client = VectorSearchWrapper(retriever_configs['retriever_1'])

# Normal search (docs don't render)
results = client.search("How do I book a flight online with Unity Airways?")

In [0]:
# Refined search (docs render)
refined_search = client.refined_search("How do I book a flight online with Unity Airways?")

## Multi Retriever

In [0]:
from mlflow.langchain.langchain_tracer import MlflowLangchainTracer
from mlflow.entities import LiveSpan, SpanEvent, SpanStatus, SpanStatusCode, SpanType
from mlflow.entities import Document as MlflowDocument
from langchain_core.messages import BaseMessage
from uuid import UUID


class CustomLangchainTracer(MlflowLangchainTracer):
    # Override the handler functions to customize the behavior. The method signature is defined by LangChain Callbacks.
    def on_chat_model_start(
        self,
        serialized: Dict[str, Any],
        messages: List[List[BaseMessage]],
        *,
        run_id: UUID,
        tags: Optional[List[str]] = None,
        parent_run_id: Optional[UUID] = None,
        metadata: Optional[Dict[str, Any]] = None,
        name: Optional[str] = None,
        **kwargs: Any,
    ):
        
        if metadata:
            kwargs.update({"reranker_metadata": metadata})

        # Call the _start_span method at the end of the handler function to start a new span.
        self._start_span(
            span_name=name or self._assign_span_name(serialized, "reranker model"),
            parent_run_id=parent_run_id,
            span_type=SpanType.RERANKER,
            run_id=run_id,
            inputs=messages,
            attributes=kwargs,
        )

In [0]:
from typing import List, Dict, Optional, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextvars
import mlflow
from mlflow.entities import SpanType
from databricks_langchain import ChatDatabricks

mlflow.langchain.autolog()


class MultiRetrieverOrchestrator:
    def __init__(
        self,
        retriever_configs: List[Dict[str, Any]],
        llm_endpoint: str = "databricks-gpt-oss-120b",
    ):
        self.retrievers = [
            VectorSearchWrapper(retriever_configs[config])
            for config in retriever_configs
        ]
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)

    def generate_retriever_queries(self, query_text: str) -> List[str]:
        """
        Use the LLM to generate a list of queries, one per retriever.
        """
        num_queries = len(self.retrievers)
        response = self.llm.invoke(
            f"Generate {num_queries} variations of this query for vector search: '{query_text}'. "
            f"Return the generated queries in a list. For example ['how are you?', 'how do you do?']"
        )

        text_items = [
            item["text"] for item in response.content if item.get("type") == "text"
        ]
        if not text_items:
            raise ValueError("LLM response does not contain any items with type='text'")

        import ast

        try:
            queries = ast.literal_eval(text_items[0])
        except Exception as e:
            raise ValueError(f"Failed to parse LLM text as list: {e}")

        if len(queries) != num_queries:
            raise ValueError(
                f"LLM returned {len(queries)} queries, expected {num_queries}"
            )

        return queries

    @mlflow.trace(span_type="Function")
    def execute_parallel_search(
        self,
        queries: List[str],
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
        refined: bool = True,
    ):
        """
        Execute retrievers in parallel using provided queries.
        """
        combined_results: List[Any] = []

        with ThreadPoolExecutor(max_workers=len(self.retrievers)) as executor:
            futures = {}
            for retriever, query in zip(self.retrievers, queries):
                ctx = contextvars.copy_context()
                search_fn = retriever.refined_search if refined else retriever.search
                futures[
                    executor.submit(
                        ctx.run, search_fn, query, columns, filters, num_results
                    )
                ] = retriever

            for future in as_completed(futures):
                try:
                    retriever_results = future.result()
                    if isinstance(retriever_results, list):
                        combined_results.extend(retriever_results)
                except Exception as e:
                    print(f"Error in retriever {futures[future]}: {e}")

        return combined_results

    def rerank_results(
        self,
        query_text: str,
        search_results: List[Document],
    ) -> List[Document]:
        """
        Use the LLM to rerank search results based on relevance to the query.

        Args:
            query_text: Original query.
            search_results: List of Document results from retrievers.

        Returns:
            List of reranked search results (full list, no truncation).
        """
        if not search_results:
            return []

        # Extract the text snippets
        docs = [res.page_content for res in search_results]

        # Construct the rerank prompt
        rerank_prompt = (
            f"Given the query:\n'{query_text}'\n\n"
            f"Rerank the following search results in order of relevance. "
            f"Return a JSON list of the ranking of each doc.\n\n"
            f"For example, [2, 3, 1] means: the first doc ranks 2nd, "
            f"the second doc ranks 3rd, and the third doc ranks 1st.\n\n"
            f"Search results:\n{docs}"
        )

        response = self.llm.invoke(
            rerank_prompt,
            config={
                "callbacks": [CustomLangchainTracer()],
                "metadata": {"input_docs": len(docs)},
                "run_name": "CustomReranker",
            },
        )

        text_items = [
            item["text"] for item in response.content if item.get("type") == "text"
        ]
        if not text_items:
            raise ValueError("LLM response did not contain any text outputs")

        import ast

        try:
            ranking = ast.literal_eval(text_items[0])
        except Exception as e:
            raise ValueError(f"Failed to parse LLM rerank output: {e}")

        if not isinstance(ranking, list) or not all(
            isinstance(x, int) for x in ranking
        ):
            raise ValueError(f"Unexpected rerank format: {ranking}")

        if len(ranking) != len(search_results):
            raise ValueError(
                f"LLM returned {len(ranking)} rankings, but {len(search_results)} documents were provided."
            )

        # Build reranked results based on ranking indices
        indexed_results = list(zip(ranking, search_results))
        indexed_results.sort(key=lambda x: x[0])  # sort by rank (1 = most relevant)
        reranked_results = [res for _, res in indexed_results]

        return reranked_results
    
    @mlflow.trace(span_type="Function")
    def apply_top_k_filter(
        self, results: List[Document], top_k: Optional[int] = None
    ) -> List[Document]:
        """
        Apply top_k truncation to results if specified.
        """
        if top_k is not None:
            return results[:top_k]
        return results

    def query_rewriting_search(
        self, query_text: str, top_k: Optional[int] = None
    ) -> List[Document]:
        """
        Orchestrate query rewriting, retrieval, and reranking, with optional top_k truncation.
        """
        with mlflow.start_span(name="Custom Retriever", span_type=SpanType.CHAIN) as span:
            span.set_inputs({"query": query_text})

            # Step 1: Generate rewritten queries
            llm_queries = self.generate_retriever_queries(query_text)

            with mlflow.start_span(name="Parallel Retrieve", span_type=SpanType.RETRIEVER) as child_span:
                # Step 2: Run parallel retrieval
                search_results = self.execute_parallel_search(llm_queries)

                # Step 3: Rerank results
                reranked_results = self.rerank_results(query_text, search_results)

                # Step 4: Apply top_k truncation after rerank
                final_results = self.apply_top_k_filter(reranked_results, top_k)
                child_span.set_outputs(final_results)

            span.set_outputs(final_results)
            return final_results
        

In [0]:
orchestrator = MultiRetrieverOrchestrator(retriever_configs)

In [0]:
results = orchestrator.query_rewriting_search(query_text="How do I book a flight online with Unity Airways?")

In [0]:

class CustomRag:
    def __init__(self, retriever, llm_endpoint: str = "databricks-gpt-oss-120b"):
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)
        self.retriever = retriever
    
    @mlflow.trace(span_type=SpanType.CHAIN, output_reducer=lambda x: "".join(x))
    def llm_call_docs(self, query_text):
        # Extract the text snippets
        docs = self.retriever.query_rewriting_search(query_text)
        docs = [doc.page_content for doc in docs]

        llm_prompt = f"""You are a trusted AI assistant that helps answer questions based only on the provided information. Here is some context which may or may not help you answer the following question: {docs}.
        
        Answer directly, do not repeat the question. Based on this context, answer this question: {query_text}.
        If you don't know the answer, just say that you don't know.
        """
        llm_stream = self.llm.stream(llm_prompt)      
        for chunk in llm_stream:
            if isinstance(chunk.content, str):
                yield chunk.content

In [0]:
custom_rag = CustomRag(orchestrator)

display_str=''
for chunk in custom_rag.llm_call_docs("How do I book a flight online with Unity Airways?"):
    display_str += chunk

display_str

## Low-Level Tracing

In [0]:
from mlflow import MlflowClient
mlflow_client = MlflowClient()

def new_search(
    self,
    query_text: str,
    columns: Optional[List[str]] = None,
    filters: Optional[Dict[str, tuple]] = None,
    num_results: Optional[int] = None,
):
    """
    Perform a similarity search against the retriever's index.

    :param query_text: The text query to search for.
    :param columns: Optional override for columns to return.
    :param filters: Optional filters to apply.
    :param num_results: Optional override for number of results (k).
    :return: Search results from the vector index.
    """
    root_span = None

    # Setup root trace
    root_span = mlflow_client.start_trace(
        name="new_single_retriever_search",
        tags={
            "vs_endpoint_name": self.retriever_cfg["endpoint_name"],
            "vs_index_name": self.retriever_cfg["index_name"],
        },
        inputs={
            "query_text": query_text,
            "columns": columns,
            "filters": filters,
            "num_results": num_results,
        },
        attributes={
            "filters": filters or self.retriever_cfg.get("filters", {}),
            "retriever_k": num_results or self.retriever_cfg.get("k", 5),
        },
    )

    # Try search
    try:
        search_results = self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )

    # Handle Search Error
    except Exception as e:
        mlflow_client.end_trace(
            request_id=root_span.request_id,
            status="ERROR",
            attributes={
                "error_type": type(e).__name__,
                "error_message": str(e),
            },
        )
        raise

    # Search success
    else:
        mlflow_client.end_trace(
            request_id=root_span.request_id,
            outputs=search_results,
            status="OK",
        )
    return search_results

In [0]:
VectorSearchWrapper.new_search = new_search
results = client.new_search("How do I book a flight online with Unity Airways?", columns=['id', 'fake_column'])

## TODO List
**Completed**
- Customize Span with Decorator (Done)
- Customize Span with Context Manager(Done)
- Trace Tags (Done)
- Trace attributes (Done)
- Combine Auto Trace (Done)
- Multi Threading (Done)
- Retriever schema (Done)
- LangChain Custom Callback (do with a reranker)
- Streaming (Complete the RAG chain)
- Low Level API: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/app-instrumentation/manual-tracing/low-level-api